# Notebook 04c: Adaptive CMF Blending Unlearning (`04c_cmf_adaptive.ipynb`)

### Objectives & Workflow:
1. **Start from Original Model & Oracle Validation Reference**: Initialize from original model $\theta_o$, reading the Oracle validation metrics from Notebook 02 as target across models (`resnet18` / `vit11m`) and seeds `[0, 1, ..., 9]`.
2. **Adaptive Head Blending**: $W_{\text{eff}} = (1 - \alpha) W_{\text{learned}} + \alpha W_{\text{CMF}}$ where $\alpha = \text{clamp}(\text{gap} / \text{scale}, 0, 1)$ and $\text{gap} = \max(\text{NCC}_f - \text{Output}_f, 0)$.
3. **Trainable Components**: Both encoder and learned head weights are optimized simultaneously.
4. **Validation-Driven Stopping**: Multi-stage loop stopping on patience or when validation $\text{NCC}_f$ reaches within tolerance of the Oracle reference on validation.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    evaluate_three_tier_metrics,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Adaptive Blending Classifier Module

In [ ]:
class AdaptiveBlendedClassifier(nn.Module):
    """
    Dynamically blends learned linear weights with class-mean features:
    W_eff = (1 - alpha) * W_learned + alpha * W_CMF
    """
    def __init__(self, encoder, num_classes=10, feature_dim=512):
        super().__init__()
        self.encoder = encoder
        self.feature_dim = feature_dim
        self.linear = nn.Linear(feature_dim, num_classes, bias=False)
        self.cmf_weights = nn.Parameter(torch.zeros(num_classes, feature_dim), requires_grad=False)
        self.alpha = 0.0
        
    def update_cmf_matrix(self, dataloader, num_classes=10, device="cuda"):
        with torch.no_grad():
            feats, targets = extract_features(self.encoder, dataloader, device=device)
            class_means = []
            for c in range(num_classes):
                mask = (targets == c)
                if mask.sum() > 0:
                    class_means.append(feats[mask].mean(axis=0))
                else:
                    class_means.append(np.zeros(feats.shape[1]))
            class_means = np.stack(class_means, axis=0)
            global_mean = class_means.mean(axis=0, keepdims=True)
            centered = class_means - global_mean
            norms = np.linalg.norm(centered, axis=1, keepdims=True) + 1e-8
            norm_weights = centered / norms
            self.cmf_weights.data.copy_(torch.tensor(norm_weights, dtype=torch.float32, device=device))
            
    def forward(self, x):
        feats = self.encoder(x)
        w_eff = (1.0 - self.alpha) * self.linear.weight + self.alpha * self.cmf_weights
        return F.linear(feats, w_eff)

## 2. Adaptive CMF Training Loop (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

STAGES = [
    {"lr": 1e-4, "max_epochs": 4, "patience": 2},
    {"lr": 3e-5, "max_epochs": 3, "patience": 2},
]
DELTA_RETAIN = 0.05
TOL_ORACLE_NCC = 0.03
MIN_DELTA = 0.002

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Dataset Loaders
forget_dataset = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)
forget_dataset_rl = copy.deepcopy(forget_dataset)
forget_dataset_rl.dataset.dataset.targets = np.random.randint(0, NUM_CLASSES, len(forget_dataset_rl.dataset.dataset.targets)).tolist()

retain_dataset = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
combined_dataset = ConcatDataset([forget_dataset_rl, retain_dataset])
train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

for seed in MODEL_SEEDS:
    kaggle_ckpt = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.pt"
    local_ckpt = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{seed}.pt"
    orig_ckpt = kaggle_ckpt if os.path.exists(kaggle_ckpt) else local_ckpt

    kaggle_metrics = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.json"
    local_metrics = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    orig_metrics_file = kaggle_metrics if os.path.exists(kaggle_metrics) else local_metrics

    oracle_metrics_file = f"./artifacts/metrics/oracle_{MODEL_NAME}_seed_{seed}.json"
    
    if not os.path.exists(orig_ckpt) or not os.path.exists(orig_metrics_file) or not os.path.exists(oracle_metrics_file):
        raise FileNotFoundError(f"Required baseline/oracle artifacts for model {MODEL_NAME} seed {seed} missing. Run Notebook 01 & 02 first.")
    print(f"Using original ckpt: {orig_ckpt} | metrics: {orig_metrics_file} | oracle: {oracle_metrics_file}")
        
    with open(orig_metrics_file, "r") as f:
        orig_val_retain = json.load(f)["validation_metrics"]["output_retain"]
    with open(oracle_metrics_file, "r") as f:
        oracle_val_ncc_f = json.load(f)["validation_reference_metrics"]["ncc_forget"]
        
    print(f"\n==================================================")
    print(f"--- Running 04c: Adaptive CMF | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"Oracle Val NCC Target: {oracle_val_ncc_f:.4f}")
    print(f"==================================================")
    
    # Load original model weights into adaptive architecture
    base_model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    base_model.load_state_dict(torch.load(orig_ckpt, map_location=device))
    
    feat_dim = base_model.encoder.feature_dim
    model = AdaptiveBlendedClassifier(base_model.encoder, num_classes=NUM_CLASSES, feature_dim=feat_dim).to(device)
    model.linear.weight.data.copy_(base_model.head.linear.weight.data)
    
    criterion = nn.CrossEntropyLoss()
    best_snapshot = None
    start_time = time.time()
    
    # Initial evaluation on validation for starting gap
    val_metrics_init = evaluate_three_tier_metrics(
        model=base_model, train_loader=train_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    
    # Initial blending alpha from initial illusion gap
    gap_ncc_init = max(val_metrics_init["ncc_forget"] - val_metrics_init["output_forget"], 0.0)
    model.alpha = float(np.clip(gap_ncc_init / 0.5, 0.0, 1.0))
    
    for stage_idx, stage in enumerate(STAGES):
        if best_snapshot is not None:
            model.load_state_dict(best_snapshot["state"])
            
        optimizer = optim.Adam([
            {"params": model.encoder.parameters(), "lr": stage["lr"]},
            {"params": model.linear.parameters(), "lr": stage["lr"] * 2}
        ])
        no_gain = 0
        
        for epoch in range(1, stage["max_epochs"] + 1):
            # Update CMF matrix
            model.update_cmf_matrix(train_loader, num_classes=NUM_CLASSES, device=device)
            
            model.train()
            for images, targets in train_loader:
                images, targets = images.to(device), targets.to(device)
                optimizer.zero_grad()
                out = model(images)
                loss = criterion(out, targets)
                loss.backward()
                optimizer.step()
                
            # Validation evaluation
            val_metrics = evaluate_three_tier_metrics(
                model=model, train_loader=train_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
            )
            
            gap_ncc = max(val_metrics["ncc_forget"] - val_metrics["output_forget"], 0.0)
            model.alpha = float(np.clip(gap_ncc / 0.5, 0.0, 1.0))
            
            # Score penalizes remaining illusion gap and rewards feature erasure + retain
            score = 0.5 * (1.0 - val_metrics["lp_forget"]) + 0.5 * (1.0 - val_metrics["ncc_forget"]) + val_metrics["output_retain"] - 0.5 * val_metrics["illusion_gap_ncc"]
            eligible = (val_metrics["output_retain"] >= orig_val_retain - DELTA_RETAIN)
            
            print(f"Stage {stage_idx+1} Ep {epoch} | Alpha: {model.alpha:.2f} | Val Retain: {val_metrics['output_retain']:.4f} | NCC Forget: {val_metrics['ncc_forget']:.4f} | Gap NCC: {val_metrics['illusion_gap_ncc']:.4f} | Score: {score:.4f}")
            
            if eligible and (best_snapshot is None or score > best_snapshot["score"] + MIN_DELTA):
                best_snapshot = {"score": score, "metrics": val_metrics, "state": copy.deepcopy(model.state_dict())}
                no_gain = 0
            else:
                no_gain += 1
                
            # Check adaptive stopping against Oracle validation NCC target
            if abs(val_metrics["ncc_forget"] - oracle_val_ncc_f) <= TOL_ORACLE_NCC:
                print(f"Reached Oracle validation NCC target ({oracle_val_ncc_f:.4f} +- {TOL_ORACLE_NCC}). Stopping early.")
                break
                
            if no_gain >= stage["patience"]:
                print(f"Early stop in stage {stage_idx+1}")
                break
                
    if best_snapshot is not None:
        model.load_state_dict(best_snapshot["state"])
        
    runtime = time.time() - start_time
    save_ckpt = f"./artifacts/checkpoints/cmf_04c_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(model.state_dict(), save_ckpt)
    
    # Final test evaluation
    test_metrics = evaluate_three_tier_metrics(
        model=model, train_loader=train_loader, retain_eval_loader=test_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    
    res = {
        "variant": "04c_cmf_adaptive",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "runtime_sec": runtime,
        "validation_metrics": best_snapshot["metrics"] if best_snapshot else val_metrics,
        "test_metrics": test_metrics
    }
    res_path = f"./artifacts/metrics/cmf_04c_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(res, f, indent=2)
        
    print(f"04c Test Retain Output Acc: {test_metrics['output_retain']:.4f} | Forget: {test_metrics['output_forget']:.4f}")
    print(f"04c Test LP Forget Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
    print(f"04c Test NCC Forget Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
    print(f"Saved 04c checkpoint & metrics to {res_path}")